In [0]:
df_bronze = spark.table("raw.default.bronze_cms_raw")

print(f"Total rows: {df_bronze.count():,}")
print(f"Total columns: {len(df_bronze.columns)}")
print("\nSchema:")
df_bronze.printSchema()

Total rows: 1,999,999
Total columns: 28

Schema:
root
 |-- Rndrng_NPI: long (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: double (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable 

In [0]:
# Cell 2 — data quality checks before cleaning
from pyspark.sql import functions as F

print("=== DATA QUALITY REPORT ===")

print(f"\n1. Rows with null NPI: {df_bronze.filter(F.col('Rndrng_NPI').isNull()).count():,}")
print(f"2. Rows with null HCPCS_Cd: {df_bronze.filter(F.col('HCPCS_Cd').isNull()).count():,}")
print(f"3. Rows with Avg_Sbmtd_Chrg < 0.01: {df_bronze.filter(F.col('Avg_Sbmtd_Chrg') < 0.01).count():,}")
print(f"4. Rows with Avg_Mdcr_Pymt_Amt == 0: {df_bronze.filter(F.col('Avg_Mdcr_Pymt_Amt') == 0).count():,}")
print(f"5. Duplicate NPI+HCPCS+Place rows: {df_bronze.count() - df_bronze.dropDuplicates(['Rndrng_NPI','HCPCS_Cd','Place_Of_Srvc']).count():,}")

=== DATA QUALITY REPORT ===

1. Rows with null NPI: 0
2. Rows with null HCPCS_Cd: 0
3. Rows with Avg_Sbmtd_Chrg < 0.01: 7
4. Rows with Avg_Mdcr_Pymt_Amt == 0: 20
5. Duplicate NPI+HCPCS+Place rows: 0


In [0]:
# Create silver schema inside raw catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS raw.silver")
spark.sql("SHOW SCHEMAS IN raw").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
|       jaffle_shop|
|            silver|
|            stripe|
+------------------+



In [0]:
# Cell 3 — clean and write Silver table
from pyspark.sql import functions as F

df_silver = df_bronze \
    .filter(F.col('Avg_Sbmtd_Chrg') >= 0.01) \
    .filter(F.col('Rndrng_NPI').isNotNull()) \
    .withColumn('Place_Of_Srvc', F.upper(F.col('Place_Of_Srvc')))

print(f"Bronze rows: {df_bronze.count():,}")
print(f"Silver rows: {df_silver.count():,}")
print(f"Rows dropped: {df_bronze.count() - df_silver.count():,}")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw.silver.cms_claims_clean")

print("\nSilver table written successfully.")

Bronze rows: 1,999,999
Silver rows: 1,999,992
Rows dropped: 7

Silver table written successfully.


In [0]:
# Cell 4 — feature engineering
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read clean silver table
df = spark.table("raw.silver.cms_claims_clean")

# Window by specialty
w_specialty = Window.partitionBy("Rndrng_Prvdr_Type")

df_features = df \
    .withColumn("charge_to_payment_ratio",
        F.when(F.col("Avg_Mdcr_Pymt_Amt") == 0, None)
         .otherwise(F.col("Avg_Sbmtd_Chrg") / F.col("Avg_Mdcr_Pymt_Amt"))) \
    .withColumn("billing_intensity",
        F.when(F.col("Tot_Benes") == 0, None)
         .otherwise(F.col("Tot_Srvcs") / F.col("Tot_Benes"))) \
    .withColumn("specialty_avg_charge",
        F.avg("Avg_Sbmtd_Chrg").over(w_specialty)) \
    .withColumn("specialty_stddev_charge",
        F.stddev("Avg_Sbmtd_Chrg").over(w_specialty)) \
    .withColumn("provider_charge_zscore",
        F.when(F.col("specialty_stddev_charge") == 0, 0)
         .otherwise(
            (F.col("Avg_Sbmtd_Chrg") - F.col("specialty_avg_charge")) /
             F.col("specialty_stddev_charge"))) \
    .withColumn("facility_flag",
        F.when(F.col("Place_Of_Srvc") == "F", 1).otherwise(0))

# Preview
df_features.select(
    "Rndrng_NPI", "Rndrng_Prvdr_Type",
    "charge_to_payment_ratio", "billing_intensity",
    "provider_charge_zscore", "facility_flag"
).show(5, truncate=False)

+----------+-----------------+-----------------------+------------------+----------------------+-------------+
|Rndrng_NPI|Rndrng_Prvdr_Type|charge_to_payment_ratio|billing_intensity |provider_charge_zscore|facility_flag|
+----------+-----------------+-----------------------+------------------+----------------------+-------------+
|1003019571|Hematology       |3.1931225053730428     |1.0               |0.04445715681912794   |0            |
|1003019571|Hematology       |3.8143070449300938     |1.0               |0.6764439927823317    |1            |
|1003019571|Hematology       |3.5646729388550886     |1.0677966101694916|-0.3979336283551147   |0            |
|1003019571|Hematology       |3.9936718009090666     |1.2142857142857142|-0.004763465005431877 |1            |
|1003019571|Hematology       |3.3343403428151674     |1.2873563218390804|-0.19371159649445682  |0            |
+----------+-----------------+-----------------------+------------------+----------------------+-------------+
o

In [0]:
# Cell 5 — write features table
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw.silver.cms_claims_features")

print(f"Features table written: {df_features.count():,} rows")
print(f"Columns: {len(df_features.columns)}")

Features table written: 1,999,992 rows
Columns: 34


In [0]:
# Cell 6 — verify both Silver tables exist
spark.sql("SHOW TABLES IN raw.silver").show()

print("\nFeature columns added:")
df_features.select(
    "charge_to_payment_ratio",
    "billing_intensity", 
    "specialty_avg_charge",
    "specialty_stddev_charge",
    "provider_charge_zscore",
    "facility_flag"
).describe().show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
|  silver|   cms_claims_clean|      false|
|  silver|cms_claims_features|      false|
+--------+-------------------+-----------+


Feature columns added:
+-------+-----------------------+------------------+--------------------+-----------------------+----------------------+-------------------+
|summary|charge_to_payment_ratio| billing_intensity|specialty_avg_charge|specialty_stddev_charge|provider_charge_zscore|      facility_flag|
+-------+-----------------------+------------------+--------------------+-----------------------+----------------------+-------------------+
|  count|                1999972|           1999992|             1999992|                1999992|               1999992|            1999992|
|   mean|      6.167977170926519|  4.79677889289573|   401.4011047664209|      733.5765222296793|  4.104459592062683...|0.36227144908579634|
| stddev|   